# Task 3 — A/B Hypothesis Testing
**AlphaCare Insurance Solutions (ACIS) | May 2026**

Statistically validate or reject four null hypotheses about risk drivers,  
forming the evidence base for ACIS's new segmentation and pricing strategy.

| # | Null Hypothesis | KPI | Test |
|---|-----------------|-----|------|
| H1 | No risk differences across provinces | Claim Severity | Welch's t-test |
| H2 | No risk differences between zip codes | Claim Severity | Welch's t-test |
| H3 | No margin difference between zip codes | Margin (ZAR) | Welch's t-test |
| H4 | No risk difference between Women and Men | Claim Frequency | z-test |

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.hypothesis_tests import (
    risk_diff_provinces,
    risk_diff_zipcodes,
    margin_diff_zipcodes,
    risk_diff_gender,
    build_results_table,
)

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')

## 1. Load Cleaned Data

In [ ]:
from src.data_loader import DataLoader

loader = DataLoader('data/processed/insurance_cleaned.csv')
df = loader.load()
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df[['TotalPremium','TotalClaims','Province','PostalCode','Gender']].head(3)

## 2. Derived Metrics Check

In [ ]:
# Ensure derived columns present
df['HasClaim']   = (df['TotalClaims'] > 0).astype(int)
df['LossRatio']  = df['TotalClaims'] / df['TotalPremium'].replace(0, np.nan)
df['Margin']     = df['TotalPremium'] - df['TotalClaims']

print(f"Total policies       : {len(df):,}")
print(f"Policies with claims : {df['HasClaim'].sum():,} ({df['HasClaim'].mean()*100:.2f}%)")
print(f"Portfolio Loss Ratio : {df['LossRatio'].mean():.4f}")
print(f"Portfolio Margin     : ZAR {df['Margin'].mean():,.2f} per policy")

## 3. H₁ — No Risk Differences Across Provinces
**KPI:** Claim Severity (mean TotalClaims where claim > 0)  
**Test:** Welch's t-test (unequal variance)  
**Groups:** Gauteng (largest province by volume) vs Western Cape (second largest)

In [ ]:
h1 = risk_diff_provinces(df, kpi='ClaimSeverity',
                          province_a='Gauteng', province_b='Western Cape')

print('=== H1 Result ===')
for k, v in h1.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Visualise: Claim Severity by Province (box plot)
claim_df = df[df['HasClaim'] == 1].copy()
top_provs = claim_df['Province'].value_counts().head(6).index

fig, ax = plt.subplots(figsize=(10, 5))
prov_order = (claim_df[claim_df['Province'].isin(top_provs)]
              .groupby('Province')['TotalClaims'].median()
              .sort_values(ascending=False).index)

sns.boxplot(data=claim_df[claim_df['Province'].isin(top_provs)],
            x='Province', y='TotalClaims', order=prov_order,
            palette='coolwarm', showfliers=False, ax=ax)

ax.set_title('Claim Severity Distribution by Province', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Claim Amount (ZAR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R {x:,.0f}'))

p_label = f"Welch's t-test p = {h1['p_value']:.4f} → {h1['decision']}"
ax.text(0.98, 0.96, p_label, transform=ax.transAxes,
        ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow', ec='gray', alpha=0.8))

plt.tight_layout()
plt.savefig('../reports/figures/h1_province_severity.png', dpi=150)
plt.show()

## 4. H₂ — No Risk Differences Between Zip Codes
**KPI:** Claim Severity  
**Test:** Welch's t-test on all pairs of the 5 highest-volume postal codes

In [ ]:
h2_results = risk_diff_zipcodes(df, kpi='ClaimSeverity', top_n=5)

h2_table = pd.DataFrame([{
    'Zip A': r['group_a'], 'Zip B': r['group_b'],
    'Mean A (ZAR)': r['mean_a'], 'Mean B (ZAR)': r['mean_b'],
    'p-value': r['p_value'], 'Decision': r['decision']
} for r in h2_results])

display(h2_table)

In [ ]:
# Visualise: Average claim by postal code
top5_codes = df['PostalCode'].value_counts().head(5).index
postal_stats = (df[df['PostalCode'].isin(top5_codes)]
                .groupby('PostalCode')
                .agg(AvgClaim=('TotalClaims','mean'),
                     AvgPremium=('TotalPremium','mean'),
                     LossRatio=('LossRatio','mean'))
                .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e74c3c' if lr > 1 else '#2ecc71' for lr in postal_stats['LossRatio']]
axes[0].bar(postal_stats['PostalCode'].astype(str), postal_stats['LossRatio'],
            color=colors, edgecolor='white')
axes[0].axhline(1.0, color='black', ls='--', lw=1.5, label='Break-even (LR=1)')
axes[0].set_title('Loss Ratio by Postal Code', fontweight='bold')
axes[0].set_ylabel('Loss Ratio')
axes[0].legend(fontsize=8)

x = np.arange(len(postal_stats))
w = 0.35
axes[1].bar(x - w/2, postal_stats['AvgPremium'], w, label='Avg Premium', color='steelblue')
axes[1].bar(x + w/2, postal_stats['AvgClaim'],   w, label='Avg Claim',   color='tomato')
axes[1].set_xticks(x)
axes[1].set_xticklabels(postal_stats['PostalCode'].astype(str))
axes[1].set_title('Avg Premium vs Avg Claim by Postal Code', fontweight='bold')
axes[1].legend()

plt.suptitle('H₂: Risk Differences Across Zip Codes', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/h2_zipcode_risk.png', dpi=150)
plt.show()

## 5. H₃ — No Margin Difference Between Zip Codes
**KPI:** Margin = TotalPremium − TotalClaims  
**Test:** Welch's t-test

In [ ]:
h3_results = margin_diff_zipcodes(df, top_n=5)

h3_table = pd.DataFrame([{
    'Zip A': r['group_a'], 'Zip B': r['group_b'],
    'Mean Margin A': r['mean_a'], 'Mean Margin B': r['mean_b'],
    'p-value': r['p_value'], 'Decision': r['decision']
} for r in h3_results])

display(h3_table)

In [ ]:
# Visualise margin distribution by postal code
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df[df['PostalCode'].isin(top5_codes)].copy()
plot_df['PostalCode'] = plot_df['PostalCode'].astype(str)

sns.violinplot(data=plot_df, x='PostalCode', y='Margin',
               palette='Set2', inner='quartile', ax=ax)
ax.axhline(0, color='red', ls='--', lw=1.5, label='Break-even (Margin=0)')
ax.set_title('H₃: Margin Distribution by Postal Code', fontsize=13, fontweight='bold')
ax.set_ylabel('Margin (ZAR)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/h3_margin_zipcodes.png', dpi=150)
plt.show()

## 6. H₄ — No Risk Difference Between Women and Men
**KPI:** Claim Frequency (proportion of policies with at least one claim)  
**Test:** Two-proportion z-test

In [ ]:
h4 = risk_diff_gender(df, kpi='ClaimFrequency')

print('=== H4 Result ===')
for k, v in h4.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Visualise: Claim frequency and severity by gender
gender_stats = (df[df['Gender'].isin(['Male','Female'])]
                .groupby('Gender')
                .agg(ClaimFreq=('HasClaim','mean'),
                     AvgSeverity=('TotalClaims', lambda x: x[x>0].mean()),
                     PolicyCount=('HasClaim','count'))
                .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

colors = ['#3498db', '#e91e8c']
axes[0].bar(gender_stats['Gender'], gender_stats['ClaimFreq']*100, color=colors, width=0.4)
axes[0].set_title('Claim Frequency by Gender', fontweight='bold')
axes[0].set_ylabel('Claim Frequency (%)')
for bar, val in zip(axes[0].patches, gender_stats['ClaimFreq']*100):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}%', ha='center', fontsize=10)

axes[1].bar(gender_stats['Gender'], gender_stats['AvgSeverity'], color=colors, width=0.4)
axes[1].set_title('Avg Claim Severity by Gender', fontweight='bold')
axes[1].set_ylabel('Avg Claim (ZAR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R {x:,.0f}'))

p_label = f"z-test p = {h4['p_value']:.4f} → {h4['decision']}"
fig.text(0.5, -0.02, p_label, ha='center', fontsize=10,
         bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

plt.suptitle('H₄: Risk Differences by Gender', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/h4_gender_risk.png', dpi=150)
plt.show()

## 7. Summary Results Table

In [ ]:
all_results = [h1] + h2_results[:1] + h3_results[:1] + [h4]
summary = build_results_table(all_results)

# Display without interpretation column for compact view
display(summary[['Hypothesis','KPI','Test','p-value','Decision']]
        .style.applymap(
            lambda v: 'background-color: #ffe0e0' if v == 'Reject H₀'
                      else ('background-color: #e0ffe0' if v == 'Fail to Reject H₀' else ''),
            subset=['Decision']
        ))

In [ ]:
print('\n=== Business Interpretations ===')
for _, row in summary.iterrows():
    print(f"\n{row['Hypothesis']}")
    print(f"  → {row['Interpretation']}")